# M3.S3 — Profiling and Performance Analysis
## Measure → locate → explain → change → remeasure
**Build M3S3-2026-10-06-v11.** Save a personal copy, then **Restart Kernel and Run All**.

We diagnose simplified real workloads: a matrix analysis pipeline, independent simulation tasks, and scientific output records. These are teaching kernels, not full production applications.

**Real experiments:** C region timers, repeated locality comparisons, GNU time, gprof, small/buffered writes, strace, OpenMP task balancing. **Conditional tools:** perf hardware counters/sampling, Score-P/Cube profiles and OTF2 traces, Valgrind memory checks. Each tool reports its actual status. **Models:** Amdahl's bound and interpretation questions.

Core runs use a **Slurm compute allocation**; Python submits jobs and analyses results. No Python-thread speedup is claimed. GPU profiling is not measured in the core lab. Optional appendices add Score-P MPI examples and a short CUDA/Nsight GPU profiling exercise.

**Live route (about 30 minutes):** Sections 1–5 → 7 → 9–11 → 16. Tool-specific extensions can be discussed from saved evidence if the queue is busy. Never invent measurements.


# 0 — Inspect the environment
**Predict:** Are the CPUs visible to Jupyter allocated to your experiment? Which tools may be restricted?


In [ ]:
import hashlib, json, math, os, platform, re, shutil, statistics, subprocess, time, uuid
from pathlib import Path
from html import escape

BUILD = "M3S3-2026-10-06-v11"
print("Build:", BUILD, "Host:", platform.node(), "Visible CPUs:", os.cpu_count())
for tool in ("sbatch", "squeue", "gcc", "gprof", "strace", "perf", "scorep", "valgrind"):
    print(tool, "→", shutil.which(tool) or "not on this host (compute node may differ)")
print("Jupyter's host is not the timing target. Declared resources matter.")
BASE_DIR = Path.cwd() / "m3s3_lab"
BASE_DIR.mkdir(exist_ok=True)
REPORT = None

def table(headers, rows):
    rows = [[str(v) for v in r] for r in rows]
    widths = [max([len(str(h))] + [len(r[i]) for r in rows]) for i, h in enumerate(headers)]
    print(" | ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in rows: print(" | ".join(v.ljust(w) for v, w in zip(r, widths)))

def bar_chart(labels, values, title, unit="seconds"):
    if not values: return
    from IPython.display import SVG, display
    top = max(values) or 1
    svg = [f'<svg xmlns="http://www.w3.org/2000/svg" width="760" height="{65+36*len(values)}">',
           '<rect width="100%" height="100%" fill="white"/>',
           f'<text x="12" y="23" font-size="17" fill="#17283c">{escape(title)} ({escape(unit)})</text>']
    for i, (label, value) in enumerate(zip(labels, values)):
        y = 48 + 36*i
        svg += [f'<text x="12" y="{y+15}" font-size="13">{escape(str(label))}</text>',
                f'<rect x="180" y="{y}" width="{420*value/top:.2f}" height="23" fill="#216e9b"/>',
                f'<text x="{190+420*value/top:.2f}" y="{y+16}" font-size="12">{value:.5g}</text>']
    display(SVG("".join(svg) + "</svg>"))

def ready():
    if REPORT is None: print("No measured report. Run Section 3 on SciTech; models and prediction questions remain usable."); return False
    return True

def summary(values):
    return (statistics.median(values), min(values), max(values))

def tool_output(name, limit=65):
    if not ready(): return
    t = REPORT["tools"].get(name, {"status":"NOT_RUN"})
    print(name, "→", t["status"])
    if "reason" in t: print(t["reason"])
    for key in ("stdout", "stderr"):
        lines = t.get(key, "").splitlines()
        if lines: print("\n" + key + ":\n" + "\n".join(lines[:limit]))
        if len(lines)>limit: print("[truncated here; full text is in report.json]")


In [ ]:
SOURCES = {'m3s3_profile_demo.c': '#define _POSIX_C_SOURCE 200809L\n#include <stdio.h>\n#include <stdlib.h>\n#include <stdint.h>\n#include <string.h>\n#include <time.h>\n#include <math.h>\n\nstatic volatile double sink_value = 0.0;\n\nstatic double now_sec(void){\n    struct timespec ts;\n    clock_gettime(CLOCK_MONOTONIC, &ts);\n    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;\n}\n\nstatic void *xmalloc_aligned(size_t align, size_t bytes){\n    void *p = NULL;\n    if (posix_memalign(&p, align, bytes) != 0 || !p) {\n        fprintf(stderr, "allocation failed for %.1f MiB\\n", bytes / 1048576.0);\n        exit(2);\n    }\n    return p;\n}\n\n__attribute__((noinline))\nstatic void initialise(double *a, size_t n){\n    for(size_t i=0;i<n;++i)\n        a[i] = 1.0 + (double)(i & 1023u) * 1e-6;\n}\n\n__attribute__((noinline))\nstatic double compute_bad(const double *a, int n, int reps){\n    volatile const double *v = a;\n    double sum = 0.0;\n    for(int r=0;r<reps;++r){\n        for(int j=0;j<n;++j){\n            for(int i=0;i<n;++i){\n                sum += v[(size_t)i*(size_t)n + (size_t)j];\n            }\n        }\n    }\n    return sum;\n}\n\n__attribute__((noinline))\nstatic double compute_good(const double *a, int n, int reps){\n    volatile const double *v = a;\n    double sum = 0.0;\n    for(int r=0;r<reps;++r){\n        for(int i=0;i<n;++i){\n            for(int j=0;j<n;++j){\n                sum += v[(size_t)i*(size_t)n + (size_t)j];\n            }\n        }\n    }\n    return sum;\n}\n\n__attribute__((noinline))\nstatic double reduction_work(const double *a, size_t n){\n    volatile const double *v = a;\n    double sum = 0.0;\n    for(size_t i=0;i<n;++i)\n        sum += v[i];\n    return sum;\n}\n\n__attribute__((noinline))\nstatic double other_work(uint64_t iters){\n    double x = 1.000001;\n    for(uint64_t i=0;i<iters;++i)\n        x = x * 1.00000001 + 0.00000003;\n    return x;\n}\n\nint main(int argc, char **argv){\n    const char *mode = (argc > 1) ? argv[1] : "bad";\n    const int n = (argc > 2) ? atoi(argv[2]) : 4096;\n    const int reps = (argc > 3) ? atoi(argv[3]) : 3;\n\n    if(strcmp(mode,"bad") != 0 && strcmp(mode,"good") != 0){\n        fprintf(stderr,"usage: %s bad|good [N] [reps]\\n", argv[0]);\n        return 1;\n    }\n\n    if(n < 16 || n > 8192 || reps < 1 || reps > 32){\n        fprintf(stderr,"N must be 16..8192; reps must be 1..32\\n"); return 2;\n    }\n    const unsigned long long other_iters = argc > 4 ? strtoull(argv[4],NULL,10) : 5000000ULL;\n    if(other_iters > 100000000ULL){fprintf(stderr,"other_iters too large\\n");return 2;}\n    size_t elems = (size_t)n * (size_t)n;\n    size_t bytes = elems * sizeof(double);\n    double *a = (double*)xmalloc_aligned(64, bytes);\n\n    double total0 = now_sec();\n\n    double t0 = now_sec();\n    initialise(a, elems);\n    double t_init = now_sec() - t0;\n\n    t0 = now_sec();\n    double c = (strcmp(mode,"bad")==0)\n        ? compute_bad(a,n,reps)\n        : compute_good(a,n,reps);\n    double t_compute = now_sec() - t0;\n\n    t0 = now_sec();\n    double r = reduction_work(a, elems);\n    double t_reduce = now_sec() - t0;\n\n    t0 = now_sec();\n    double o = other_work(other_iters);\n    double t_other = now_sec() - t0;\n\n    double total = now_sec() - total0;\n    sink_value += c + r + o;\n    /* Independent reference for the repeating 1024-value initialization. */\n    size_t blocks=elems/1024, tail=elems%1024;\n    long double reference=(long double)elems + 1e-6L *\n        ((long double)blocks*1023*1024/2 + (long double)tail*(tail-1)/2);\n    if(!isfinite(c) || !isfinite(r) || !isfinite(o) ||\n       fabsl((long double)c-reference*reps)>1e-8L*fmaxl(1,reference*reps) ||\n       fabsl((long double)r-reference)>1e-8L*fmaxl(1,reference)){\n        fprintf(stderr,"VALIDATION_FAIL matrix_sum\\n"); free(a); return 3;\n    }\n    printf("VALIDATION name=matrix_sum result=PASS\\n");\n\n    printf("PROFILE_META mode=%s n=%d reps=%d other_iters=%llu matrix_MiB=%.1f\\n",\n           mode,n,reps,other_iters,bytes/1048576.0);\n    printf("PROFILE_PHASE mode=%s phase=initialise seconds=%.6f\\n",mode,t_init);\n    printf("PROFILE_PHASE mode=%s phase=compute seconds=%.6f\\n",mode,t_compute);\n    printf("PROFILE_PHASE mode=%s phase=reduction seconds=%.6f\\n",mode,t_reduce);\n    printf("PROFILE_PHASE mode=%s phase=other seconds=%.6f\\n",mode,t_other);\n    printf("PROFILE_TOTAL mode=%s seconds=%.6f checksum=%.12e\\n",mode,total,c+r);\n    fprintf(stderr,"sink=%f\\n",sink_value);\n\n    free(a);\n    return 0;\n}\n\n', 'm3s3_openmp_wait.c': '#include <omp.h>\n#include <stdint.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <math.h>\n\n__attribute__((noinline))\nstatic double do_work(uint64_t iters){\n    double x=1.000001;\n    for(uint64_t i=0;i<iters;++i) x=x*1.00000001+0.00000003;\n    return x;\n}\nint main(int argc,char **argv){\n    const char *mode=argc>1?argv[1]:"skewed";\n    int requested=argc>2?atoi(argv[2]):4, skew=argc>3?atoi(argv[3]):8;\n    uint64_t base=argc>4?strtoull(argv[4],NULL,10):2000000;\n    if((strcmp(mode,"skewed")&&strcmp(mode,"balanced")) || requested<1 || requested>64 ||\n       skew<1 || skew>32 || base<100 || base>10000000){fprintf(stderr,"invalid OpenMP settings\\n");return 2;}\n    int tasks=requested-1+skew, actual=0;\n    double compute[64]={0}, wait[64]={0}, start[64]={0}, end[64]={0}, results[64]={0};\n    int counts[64]={0};\n    omp_set_dynamic(0);omp_set_num_threads(requested);\n    double origin=0, wall0=omp_get_wtime();\n    #pragma omp parallel shared(origin,actual)\n    {\n        int tid=omp_get_thread_num();\n        #pragma omp single\n        {actual=omp_get_num_threads();origin=omp_get_wtime();}\n        start[tid]=omp_get_wtime()-origin;\n        for(int task=0;task<tasks;++task){\n            int owner=!strcmp(mode,"balanced")?task%actual:(task<actual-1?task:actual-1);\n            if(owner==tid){results[tid]+=do_work(base);counts[tid]++;}\n        }\n        end[tid]=omp_get_wtime()-origin;\n        compute[tid]=end[tid]-start[tid];\n        double t0=omp_get_wtime();\n        #pragma omp barrier\n        wait[tid]=omp_get_wtime()-t0;\n    }\n    double wall=omp_get_wtime()-wall0, sum=0;int done=0;\n    for(int i=0;i<actual;++i){sum+=results[i];done+=counts[i];}\n    double ref=tasks*do_work(base);\n    if(actual!=requested || done!=tasks || fabs(sum-ref)>1e-10*fmax(1,fabs(ref))){\n        fprintf(stderr,"VALIDATION_FAIL parallel_task_ownership actual=%d requested=%d\\n",actual,requested);return 3;\n    }\n    for(int i=0;i<actual;++i)\n        printf("OMP_TRACE mode=%s thread=%d threads=%d tasks=%d work_iters=%llu start=%.9f compute_end=%.9f compute=%.9f barrier_wait=%.9f\\n",\n        mode,i,actual,counts[i],(unsigned long long)(base*counts[i]),start[i],end[i],compute[i],wait[i]);\n    printf("OMP_TOTAL mode=%s tasks=%d seconds=%.9f checksum=%.12e\\n",mode,tasks,wall,sum);\n    printf("VALIDATION name=parallel_task_ownership result=PASS\\n");return 0;\n}\n', 'm3s3_syscall_demo.c': '#define _POSIX_C_SOURCE 200809L\n#include <errno.h>\n#include <fcntl.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <stdint.h>\n#include <time.h>\n#include <unistd.h>\nstatic double now(void){struct timespec t;clock_gettime(CLOCK_MONOTONIC,&t);return t.tv_sec+t.tv_nsec*1e-9;}\nint main(int argc,char **argv){\n    const char *mode=argc>1?argv[1]:"small";\n    int records=argc>2?atoi(argv[2]):4000, chunk_records=argc>3?atoi(argv[3]):256;\n    if((strcmp(mode,"small")&&strcmp(mode,"buffered"))||records<1||records>100000||chunk_records<1||chunk_records>4096)return 2;\n    char path[]="m3s3_io_XXXXXX";int fd=mkstemp(path);if(fd<0){perror("mkstemp");return 2;}\n    size_t total=(size_t)records*64, chunk=!strcmp(mode,"small")?64:(size_t)chunk_records*64;\n    char *buf=malloc(chunk);if(!buf){close(fd);unlink(path);return 2;}memset(buf,\'A\',chunk);\n    int calls=0;size_t pos=0;double t0=now();\n    while(pos<total){size_t target=total-pos<chunk?total-pos:chunk, off=0;\n        while(off<target){ssize_t w=write(fd,buf+off,target-off);calls++;\n            if(w<0&&errno==EINTR)continue;\n            if(w<=0){perror("write");free(buf);close(fd);unlink(path);return 3;}off+=(size_t)w;}\n        pos+=target;\n    }\n    if(close(fd)){perror("close");free(buf);unlink(path);return 3;}\n    double secs=now()-t0;\n    fd=open(path,O_RDONLY);if(fd<0){free(buf);unlink(path);return 3;}\n    uint64_t checksum=0;size_t read_bytes=0;int ok=1;ssize_t got;\n    while((got=read(fd,buf,chunk))>0){for(ssize_t i=0;i<got;++i){if(buf[i]!=\'A\')ok=0;checksum+=(unsigned char)buf[i];}read_bytes+=(size_t)got;}\n    if(got<0)ok=0;\n    close(fd);unlink(path);free(buf);\n    if(!ok||read_bytes!=total||checksum!=65ULL*total){fprintf(stderr,"VALIDATION_FAIL io_bytes\\n");return 3;}\n    printf("IO mode=%s records=%d bytes=%zu chunk_records=%d write_calls=%d seconds=%.9f checksum=%llu\\n",mode,records,total,chunk_records,calls,secs,(unsigned long long)checksum);\n    printf("VALIDATION name=io_bytes result=PASS\\n");return 0;\n}\n', 'm3s3_memory_check.c': '#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n__attribute__((noinline)) static void allocation(int leak){\n    int *a=malloc(4096*sizeof(int));if(!a)exit(2);\n    for(int i=0;i<4096;++i)a[i]=i;\n    printf("MEMORY checksum=%d\\n",a[1]+a[4095]);\n    if(!leak)free(a); /* Intentional leak only in the named teaching mode. */\n}\nint main(int argc,char **argv){\n    if(argc!=2||(strcmp(argv[1],"leak")&&strcmp(argv[1],"fixed")))return 2;\n    allocation(!strcmp(argv[1],"leak"));return 0;\n}\n', 'm3s3_run_lab.py': '#!/usr/bin/env python3\n"""Real measurements; standard library only. Called on the allocated compute node."""\nimport hashlib, json, math, os, platform, shutil, subprocess, sys, time\nfrom pathlib import Path\n\nROOT=Path(__file__).resolve().parent\nos.chdir(ROOT)\nS=json.loads(Path(sys.argv[1] if len(sys.argv)>1 else \'settings.json\').read_text())\nREPORT=Path(\'report.json\')\nFLAGS=[\'-O3\',\'-g\',\'-fno-omit-frame-pointer\',\'-std=c11\']\nCPUS=sorted(os.sched_getaffinity(0)) if hasattr(os,\'sched_getaffinity\') else []\nR={\'schema\':1,\'settings\':S,\'host\':platform.node(),\'job_id\':os.environ.get(\'SLURM_JOB_ID\'),\n   \'compiler_flags\':FLAGS,\'allowed_cpus\':CPUS,\'measurements\':{},\'tools\':{},\'artifacts\':{},\n   \'source_sha256\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in ROOT.glob(\'*.c\')},\'core_status\':\'RUNNING\'}\n\ndef save():\n    Path(\'report.json.tmp\').write_text(json.dumps(R,indent=2));Path(\'report.json.tmp\').replace(REPORT)\n\ndef command(argv, *, serial=True, env=None, timeout=90):\n    child_env=os.environ.copy();child_env.update({\'LC_ALL\':\'C\',\'OMP_DYNAMIC\':\'FALSE\',\'OMP_PROC_BIND\':\'close\',\'OMP_PLACES\':\'cores\'})\n    if env:child_env.update(env)\n    print(\'$ \'+\' \'.join(map(str,argv)),flush=True)\n    pin=(lambda:os.sched_setaffinity(0,{CPUS[0]})) if serial and CPUS else None\n    t=time.perf_counter()\n    try:\n        p=subprocess.run(list(map(str,argv)),cwd=ROOT,env=child_env,capture_output=True,text=True,\n                         timeout=timeout,preexec_fn=pin)\n        result={\'returncode\':p.returncode,\'stdout\':p.stdout,\'stderr\':p.stderr,\'process_wall\':time.perf_counter()-t}\n    except (subprocess.TimeoutExpired,OSError) as exc:\n        result={\'returncode\':124,\'stdout\':\'\',\'stderr\':str(exc),\'process_wall\':time.perf_counter()-t}\n    print(result[\'stdout\'],end=\'\',flush=True)\n    if result[\'stderr\']:print(result[\'stderr\'],file=sys.stderr,flush=True)\n    return result\n\ndef core(argv,**kw):\n    p=command(argv,**kw)\n    if p[\'returncode\']!=0:raise RuntimeError(f\'Core command failed ({p["returncode"]}): {argv}\')\n    return p\n\ndef kv(text,prefix):\n    return [dict(t.split(\'=\',1) for t in line.split()[1:] if \'=\' in t)\n            for line in text.splitlines() if line.startswith(prefix+\' \')]\n\ndef checked(argv,prefix,**kw):\n    p=core(argv,**kw);rows=kv(p[\'stdout\'],prefix)\n    if \'result=PASS\' not in p[\'stdout\'] or not rows:raise RuntimeError(\'Missing validation / experiment output\')\n    for row in rows:\n        for key in (\'seconds\',\'compute\',\'barrier_wait\'):\n            if key in row and (not math.isfinite(float(row[key])) or float(row[key])<0):raise RuntimeError(\'Invalid timing\')\n    return p,rows\n\ndef optional(name, executable, argv, *, serial=True, artifact=None, env=None, timeout=90):\n    if not shutil.which(executable):R[\'tools\'][name]={\'status\':\'UNAVAILABLE\',\'reason\':f\'{executable} not found\'};return None\n    p=command(argv,serial=serial,env=env,timeout=timeout)\n    status=\'OK\' if p[\'returncode\']==0 else \'FAILED_OR_RESTRICTED\'\n    if status==\'OK\' and artifact and not (ROOT/artifact).exists():status=\'FAILED_OR_RESTRICTED\'\n    R[\'tools\'][name]={\'status\':status,**p}\n    if artifact and (ROOT/artifact).exists():R[\'artifacts\'][name]=artifact\n    return p\n\ndef main():\n    if S[\'THREADS\']>len(CPUS) and CPUS:raise RuntimeError(\'THREADS exceeds CPUs permitted by the allocation\')\n    compiler=shutil.which(\'gcc\')\n    if not compiler:raise RuntimeError(\'gcc is required for the core lab\')\n    R[\'compiler_version\']=core([compiler,\'--version\'])[\'stdout\'].splitlines()[0]\n    specs=[(\'profile\',\'m3s3_profile_demo.c\',[]),(\'profile_pg\',\'m3s3_profile_demo.c\',[\'-pg\']),\n           (\'omp_wait\',\'m3s3_openmp_wait.c\',[\'-fopenmp\']),(\'io_demo\',\'m3s3_syscall_demo.c\',[]),\n           (\'memory_check\',\'m3s3_memory_check.c\',[])]\n    for binary,source,extra in specs:core([compiler,*FLAGS,*extra,source,\'-lm\',\'-o\',binary])\n    def profile(mode,binary=\'profile\'):return [\'./\'+binary,mode,str(S[\'N\']),str(S[\'REPS\']),str(S[\'OTHER_ITERS\'])]\n    # Warm up both modes, then alternate A/B order. Never mix profiled timings into this baseline.\n    for mode in (\'bad\',\'good\'):checked(profile(mode),\'PROFILE_TOTAL\')\n    samples=[]\n    for rep in range(S[\'RUNS\']):\n        for mode in ((\'bad\',\'good\') if rep%2==0 else (\'good\',\'bad\')):\n            p,total=checked(profile(mode),\'PROFILE_TOTAL\')\n            samples.append({\'run\':rep+1,\'mode\':mode,\'total\':total[0],\n                            \'phases\':kv(p[\'stdout\'],\'PROFILE_PHASE\'),\'process_wall\':p[\'process_wall\']})\n    checksums=[float(x[\'total\'][\'checksum\']) for x in samples]\n    if max(checksums)-min(checksums)>1e-8*max(1,max(checksums)):raise RuntimeError(\'Locality modes disagree\')\n    R[\'measurements\'][\'locality\']=samples\n    omp=[];io=[]\n    def omp_args(mode):return [\'./omp_wait\',mode,str(S[\'THREADS\']),str(S[\'SKEW\']),str(S[\'WORK_ITERS\'])]\n    def io_args(mode):return [\'./io_demo\',mode,str(S[\'RECORDS\']),str(S[\'BUFFER_RECORDS\'])]\n    for mode in (\'skewed\',\'balanced\'):checked(omp_args(mode),\'OMP_TOTAL\',serial=False)\n    for mode in (\'small\',\'buffered\'):checked(io_args(mode),\'IO\')\n    for rep in range(S[\'RUNS\']):\n        for mode in ((\'skewed\',\'balanced\') if rep%2==0 else (\'balanced\',\'skewed\')):\n            p,rows=checked(omp_args(mode),\'OMP_TOTAL\',serial=False)\n            omp.append({\'run\':rep+1,\'mode\':mode,\'total\':rows[0],\'threads\':kv(p[\'stdout\'],\'OMP_TRACE\')})\n        for mode in ((\'small\',\'buffered\') if rep%2==0 else (\'buffered\',\'small\')):\n            p,rows=checked(io_args(mode),\'IO\');io.append({\'run\':rep+1,**rows[0]})\n    if len({x[\'total\'][\'checksum\'] for x in omp})!=1:raise RuntimeError(\'Parallel modes disagree\')\n    if len({(x[\'bytes\'],x[\'checksum\']) for x in io})!=1:raise RuntimeError(\'I/O modes disagree\')\n    R[\'measurements\'][\'openmp\']=omp;R[\'measurements\'][\'io\']=io\n    if Path(\'/usr/bin/time\').exists():optional(\'gnu_time\',\'/usr/bin/time\',[\'/usr/bin/time\',\'-v\',*profile(\'bad\')])\n    else:R[\'tools\'][\'gnu_time\']={\'status\':\'UNAVAILABLE\',\'reason\':\'/usr/bin/time missing\'}\n    # Instrumentation overhead: same flags and workload; paired order; raw measurements retained.\n    pg=[]\n    for rep in range(S[\'RUNS\']):\n        for binary in ((\'profile\',\'profile_pg\') if rep%2==0 else (\'profile_pg\',\'profile\')):\n            p,rows=checked(profile(\'bad\',binary),\'PROFILE_TOTAL\')\n            pg.append({\'run\':rep+1,\'binary\':binary,\'seconds\':rows[0][\'seconds\'],\'process_wall\':p[\'process_wall\']})\n    R[\'measurements\'][\'instrumentation\']=pg\n    gp=optional(\'gprof\',\'gprof\',[\'gprof\',\'./profile_pg\',\'gmon.out\'])\n    if gp and gp[\'returncode\']==0 and \'no time accumulated\' in gp[\'stdout\'].lower():\n        R[\'tools\'][\'gprof\'][\'status\']=\'NO_SAMPLES\'\n    for mode in (\'small\',\'buffered\'):\n        optional(\'strace_\'+mode,\'strace\',[\'strace\',\'-c\',*io_args(mode)])\n    optional(\'strace_events\',\'strace\',[\'strace\',\'-tt\',\'-T\',\'-e\',\'trace=openat,write,close,unlink\',\'-o\',\'strace_events.txt\',*io_args(\'buffered\')],artifact=\'strace_events.txt\')\n    if S[\'PERF\']:\n        for mode in (\'bad\',\'good\'):\n            result=optional(\'perf_stat_\'+mode,\'perf\',[\'perf\',\'stat\',\'-x\',\';\',\'-e\',\'cycles,instructions,cache-references,cache-misses\',*profile(mode)])\n            if result and any(w in result[\'stderr\'] for w in (\'<not supported>\',\'<not counted>\')):\n                R[\'tools\'][\'perf_stat_\'+mode][\'status\']=\'PARTIAL_COUNTERS\'\n        optional(\'perf_sampling\',\'perf\',[\'perf\',\'record\',\'-g\',\'-o\',\'perf.data\',\'--\',*profile(\'bad\')],artifact=\'perf.data\')\n        if R[\'tools\'][\'perf_sampling\'][\'status\']==\'OK\':optional(\'perf_report\',\'perf\',[\'perf\',\'report\',\'--stdio\',\'--no-children\',\'-i\',\'perf.data\'])\n    else:R[\'tools\'][\'perf_stat_bad\']={\'status\':\'DISABLED\',\'reason\':\'PERF=False\'}\n    if S[\'MEMORY_CHECK\']:\n        for mode in (\'leak\',\'fixed\'):\n            p=optional(\'valgrind_\'+mode,\'valgrind\',[\'valgrind\',\'--leak-check=full\',\'--show-leak-kinds=definite\',\'--errors-for-leak-kinds=definite\',\'--error-exitcode=23\',\'./memory_check\',mode])\n            if p:\n                expected=23 if mode==\'leak\' else 0\n                verified=p[\'returncode\']==expected and (\'definitely lost: 16,384 bytes\' in p[\'stderr\'] if mode==\'leak\' else \'All heap blocks were freed\' in p[\'stderr\'])\n                R[\'tools\'][\'valgrind_\'+mode][\'status\']=\'EXPECTED_LEAK_DETECTED\' if mode==\'leak\' and verified else (\'OK\' if verified else \'FAILED_OR_RESTRICTED\')\n    else:R[\'tools\'][\'valgrind\']={\'status\':\'DISABLED\',\'reason\':\'MEMORY_CHECK=False\'}\n    if S[\'SCOREP\']:\n        if not shutil.which(\'scorep\'):\n            R[\'tools\'][\'scorep\']={\'status\':\'UNAVAILABLE\',\'reason\':\'Score-P module not available / not loaded\'}\n        else:\n            for name,source,extra in [(\'serial\',\'m3s3_profile_demo.c\',[]),(\'openmp\',\'m3s3_openmp_wait.c\',[\'-fopenmp\'])]:\n                build=optional(\'scorep_build_\'+name,\'scorep\',[\'scorep\',compiler,*FLAGS,*extra,source,\'-lm\',\'-o\',\'scorep_\'+name])\n                if not build or build[\'returncode\']:continue\n                argv=[\'./scorep_serial\',*profile(\'bad\')[2:]] if name==\'serial\' else [\'./scorep_openmp\',*omp_args(\'skewed\')[1:]]\n                if name==\'serial\':argv.insert(1,\'bad\')\n                directory=\'scorep_\'+name+\'_profile\'\n                optional(\'scorep_profile_\'+name,\'scorep\',argv,serial=name==\'serial\',env={\'SCOREP_EXPERIMENT_DIRECTORY\':str(ROOT/directory),\'SCOREP_OVERWRITE_EXPERIMENT_DIRECTORY\':\'true\',\'SCOREP_ENABLE_PROFILING\':\'true\',\'SCOREP_ENABLE_TRACING\':\'false\'},artifact=directory+\'/profile.cubex\')\n                if R[\'tools\'][\'scorep_profile_\'+name][\'status\']==\'OK\':\n                    optional(\'scorep_score_\'+name,\'scorep-score\',[\'scorep-score\',directory+\'/profile.cubex\'])\n                    optional(\'cube_\'+name,\'cube_dump\',[\'cube_dump\',\'-m\',\'time\',\'-t\',\'aggr\',\'-z\',\'incl\',\'-s\',\'human\',directory+\'/profile.cubex\'])\n            if R[\'tools\'].get(\'scorep_build_openmp\',{}).get(\'status\')==\'OK\':\n                directory=\'scorep_openmp_trace\'\n                optional(\'scorep_trace\',\'scorep\',[\'./scorep_openmp\',*omp_args(\'skewed\')[1:]],serial=False,env={\'SCOREP_EXPERIMENT_DIRECTORY\':str(ROOT/directory),\'SCOREP_OVERWRITE_EXPERIMENT_DIRECTORY\':\'true\',\'SCOREP_ENABLE_PROFILING\':\'false\',\'SCOREP_ENABLE_TRACING\':\'true\',\'SCOREP_TOTAL_MEMORY\':\'64M\'},artifact=directory+\'/traces.otf2\')\n    else:R[\'tools\'][\'scorep\']={\'status\':\'DISABLED\',\'reason\':\'SCOREP=False\'}\n    R[\'core_status\']=\'PASS\';save()\n    print(\'M3S3_CORE_RESULT=PASS\',flush=True)\n    for name,tool in R[\'tools\'].items():print(f\'TOOL_STATUS name={name} status={tool["status"]}\',flush=True)\n\nif __name__==\'__main__\':\n    try:main()\n    except Exception as exc:\n        R[\'core_status\']=\'FAIL\';R[\'error\']=str(exc);save();print(\'M3S3_CORE_RESULT=FAIL \'+str(exc),file=sys.stderr);sys.exit(1)\n', 'm3s3_profile_lab.sbatch': '#!/bin/bash\n#SBATCH --job-name=m3s3_profile\n#SBATCH --partition=cpu\n#SBATCH --nodes=1\n#SBATCH --ntasks=1\n#SBATCH --cpus-per-task=4\n#SBATCH --mem=4G\n#SBATCH --time=00:12:00\n#SBATCH --output=m3s3_profile-%j.out\nset -e -o pipefail\ncd "${SLURM_SUBMIT_DIR:-$(dirname "$0")}"\nPYTHON_BIN=$(command -v python3)\nexport OMP_DYNAMIC=FALSE OMP_PROC_BIND=close OMP_PLACES=cores\n# Core compiler availability first; profiling modules are optional.\nEESSI_INIT=/cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash\nif [ -r "$EESSI_INIT" ]; then\n    source "$EESSI_INIT"\n    module load foss/2023b || true\n    module load Score-P/8.4-gompi-2023b || true\n    module load CubeLib/4.8.2-GCCcore-13.2.0 || true\nfi\nprintf \'M3S3_JOB job=%s host=%s cpus=%s\\n\' "${SLURM_JOB_ID:-none}" "$(hostname)" "${SLURM_CPUS_PER_TASK:-unknown}"\nlscpu\n# The Python runner and all child tools execute inside the same job step.\nsrun --ntasks=1 --cpus-per-task="${SLURM_CPUS_PER_TASK:-4}" --cpu-bind=cores "$PYTHON_BIN" m3s3_run_lab.py settings.json\n'}

def show_source(name, function=None):
    text = SOURCES[name]
    if function:
        match = re.search(r"(?m)^.*\b" + re.escape(function) + r"\([^;]*?\)\s*\{", text)
        if not match: raise ValueError("Function not found: " + function)
        first = text.find("{", match.start()); depth = 1; end = first+1
        while depth and end<len(text):
            if text[end]=="{": depth+=1
            elif text[end]=="}": depth-=1
            end+=1
        text = text[match.start():end]
    print(text)
print("Sources embedded in this notebook:", ", ".join(SOURCES))
print("No external downloads or pip installs are required.")


# 1 — Change one setting, predict, then run
Default workload: **4096×4096 doubles (128 MiB)**, 3 traversals, 5 repeated pairs, 4 OpenMP threads.

Choose ONE experiment:

| Change | Prediction to test |
|---|---|
| `N`: 4096 → 2048 | Does the locality ratio change as the matrix shrinks? |
| `OTHER_ITERS`: 5,000,000 → 40,000,000 | Does another phase become the limiting cost? |
| `SKEW`: 8 → 1 | Does barrier waiting disappear when original ownership is balanced? |
| `THREADS`: 4 → 2 | Does balanced work still finish faster? |
| `BUFFER_RECORDS`: 256 → 1 | Does buffered mode become equivalent to small writes? |
| `RECORDS`: 4000 → 16000 | How do write counts and runtime change? |

Keep the other settings fixed. **RUNS** controls measurement repetitions; **REPS** controls work inside the matrix program. They are different.


In [ ]:
LAB_SETTINGS = {
    "N": 4096, "REPS": 3, "RUNS": 5, "OTHER_ITERS": 5_000_000,
    "THREADS": 4, "SKEW": 8, "WORK_ITERS": 2_000_000,
    "RECORDS": 4000, "BUFFER_RECORDS": 256,
    "PERF": True, "SCOREP": True, "MEMORY_CHECK": True,
}
# Use SciTech defaults for class. Optional tools can be disabled to shorten an experiment.
RUN_LOCAL = False  # Only for your own machine without Slurm; local timings are not cluster evidence.
REPEAT_SAME_EXPERIMENT = False  # True submits one new repeat; otherwise completed jobs are reused.
WAIT_SECONDS = 600
PARTITION = "cpu"
BOUNDS = {"N":(16,8192), "REPS":(1,32), "RUNS":(3,9), "OTHER_ITERS":(0,100_000_000),
          "THREADS":(1,16), "SKEW":(1,32), "WORK_ITERS":(100,10_000_000),
          "RECORDS":(1,100_000), "BUFFER_RECORDS":(1,4096)}
for name, (low, high) in BOUNDS.items():
    v=LAB_SETTINGS[name]
    if type(v) is not int or not low<=v<=high: raise ValueError(f"{name} must be an integer in {low}..{high}")
for name in ("PERF", "SCOREP", "MEMORY_CHECK"):
    if type(LAB_SETTINGS[name]) is not bool: raise ValueError(name + " must be Boolean")
print(json.dumps(LAB_SETTINGS, indent=2))
print(f"Matrix: {LAB_SETTINGS['N']**2*8/2**20:.1f} MiB. Slurm: {LAB_SETTINGS['THREADS']} CPUs, 4 GiB, 12 min.")


# 2 — See the actual C computation
Both loops sum all matrix values, but their inner loops visit different addresses. C stores these rows contiguously.

A climate/image pipeline might have this access mismatch. **Volatile reads are deliberate here** to retain a transparent scalar access pattern; production kernels should also investigate compiler vectorization.

Initialization, summation, reduction and independent scalar work are timed separately. Every run checks its matrix sums against an independently calculated reference, allowing floating-point rounding tolerance.


In [ ]:
SHOW_C_CODE = True
if SHOW_C_CODE:
    for fn in ("compute_bad", "compute_good"):
        show_source("m3s3_profile_demo.c", fn)


# 3 — Run or reuse the real experiment
The notebook writes its embedded sources into a private run directory. Completed results survive kernel restarts. The fingerprint includes **settings and sources**. Changing either produces a new experiment; an active job must finish before another starts.

Warmups are excluded. A/B order alternates between repetitions to reduce order bias. All serial measurements are pinned to one CPU within the allocation; the OpenMP experiment uses the allocated CPU set. Runs are sequential, never simultaneous.

If the queue exceeds the wait limit, this cell stops. Rerun **this cell** later to collect the same job, then Run All Below. Do not resubmit just because a job is pending.


In [ ]:
def clean_slurm_env():
    env=os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key,None)
    return env

def queue_state(job):
    p=subprocess.run(["squeue","-h","-j",str(job),"-o","%T"],capture_output=True,text=True)
    if p.returncode:
        if "Invalid job id specified" in p.stderr:
            return ""
        raise RuntimeError("Cannot check queue: " + p.stderr)
    return p.stdout.strip()

def collect(state):
    global REPORT, RUN_DIR, M3S3_JOB_ID
    RUN_DIR=Path(state["run_dir"]); M3S3_JOB_ID=state["job_id"]
    path=RUN_DIR/"report.json"
    start=time.monotonic()
    while True:
        if path.exists():
            report=json.loads(path.read_text())
            if report["core_status"]=="FAIL": raise RuntimeError("Lab failed: " + report.get("error","see job log"))
            if report["core_status"]=="PASS":
                if report["settings"]!=LAB_SETTINGS: raise RuntimeError("Settings/report mismatch")
                actual={n:hashlib.sha256(SOURCES[n].encode()).hexdigest() for n in SOURCES if n.endswith('.c')}
                if actual!=report["source_sha256"]: raise RuntimeError("Source/report mismatch")
                REPORT=report;break
        status=queue_state(M3S3_JOB_ID)
        if not status:
            p=subprocess.run(["sacct","-n","-j",str(M3S3_JOB_ID),"--format=State,ExitCode"],capture_output=True,text=True)
            if any(w in p.stdout for w in ("FAILED","CANCELLED","TIMEOUT","OUT_OF_MEMORY","NODE_FAIL")):
                raise RuntimeError("Job failed: " + p.stdout + "\nSee " + str(RUN_DIR/f"m3s3_profile-{M3S3_JOB_ID}.out"))
        elapsed=time.monotonic()-start
        if elapsed>=WAIT_SECONDS:
            raise RuntimeError(f"Job {M3S3_JOB_ID} has no completed report yet. Rerun THIS cell later to reuse it. Log: {RUN_DIR}")
        if int(elapsed)%30<5: print(f"Job {M3S3_JOB_ID}: {status or 'waiting for filesystem/report'}; {elapsed:.0f}s",flush=True)
        time.sleep(5)
    record={"job_id":M3S3_JOB_ID,"run_dir":str(RUN_DIR),"settings":LAB_SETTINGS,
            "source_fingerprint":state["fingerprint"],"report":REPORT}
    (BASE_DIR/f"experiment-{M3S3_JOB_ID}.json").write_text(json.dumps(record,indent=2))
    print("CORE PASS — correctness and measurements complete. Host:",REPORT["host"])
    print("Allowed CPUs:",REPORT["allowed_cpus"])
    table(["Tool", "Actual status"],[(n,t["status"]) for n,t in REPORT["tools"].items()])
    print("Full report:",path)
    print("Experiment ID:",M3S3_JOB_ID)

fingerprint=hashlib.sha256(json.dumps({"settings":LAB_SETTINGS,"sources":SOURCES},sort_keys=True).encode()).hexdigest()
state_path=BASE_DIR/"last-job.json"
state=json.loads(state_path.read_text()) if state_path.exists() else None
HAVE_SLURM=bool(shutil.which("sbatch") and shutil.which("squeue"))
if HAVE_SLURM:
    state_status = queue_state(state["job_id"]) if state else ""
    if state and state_status:
        if state["fingerprint"]!=fingerprint or REPEAT_SAME_EXPERIMENT:
            raise RuntimeError(f"Job {state['job_id']} is active. Restore its settings and collect it before starting another experiment.")

    if state and not state_status:
        saved_report = Path(state["run_dir"]) / "report.json"
        if not saved_report.exists():
            print(f"Saved job {state['job_id']} is no longer known to Slurm and has no report. Starting a fresh run.")
            state = None
            state_path.unlink(missing_ok=True)

    if state and state["fingerprint"]==fingerprint and not REPEAT_SAME_EXPERIMENT:
        print("Reusing saved job",state["job_id"])
    else:
        RUN_DIR=BASE_DIR/("run-"+uuid.uuid4().hex[:12]);RUN_DIR.mkdir()
        for name,source in SOURCES.items(): (RUN_DIR/name).write_text(source)
        (RUN_DIR/"settings.json").write_text(json.dumps(LAB_SETTINGS))
        p=subprocess.run(["sbatch","--parsable","--partition="+PARTITION,
                          "--cpus-per-task="+str(LAB_SETTINGS["THREADS"]),"m3s3_profile_lab.sbatch"],
                         cwd=RUN_DIR,capture_output=True,text=True,env=clean_slurm_env())
        if p.returncode: raise RuntimeError("Submission failed: " + p.stderr)
        job=p.stdout.strip().split(';')[0]
        if not job.isdigit(): raise RuntimeError("Unexpected job ID: " + p.stdout)
        state={"job_id":job,"fingerprint":fingerprint,"run_dir":str(RUN_DIR)}
        state_path.write_text(json.dumps(state));print("Submitted and saved",job)
        REPEAT_SAME_EXPERIMENT=False
    collect(state)
elif RUN_LOCAL:
    RUN_DIR=BASE_DIR/("local-"+uuid.uuid4().hex[:12]);RUN_DIR.mkdir()
    for name,source in SOURCES.items(): (RUN_DIR/name).write_text(source)
    (RUN_DIR/"settings.json").write_text(json.dumps(LAB_SETTINGS))
    import sys
    with (RUN_DIR/"local.log").open('w') as log:
        p=subprocess.run([sys.executable,"m3s3_run_lab.py","settings.json"],cwd=RUN_DIR,stdout=log,stderr=subprocess.STDOUT,timeout=600)
    REPORT=json.loads((RUN_DIR/"report.json").read_text())
    if p.returncode or REPORT["core_status"]!="PASS": raise RuntimeError("Local lab failed; inspect " + str(RUN_DIR/"local.log"))
    M3S3_JOB_ID=RUN_DIR.name
    (BASE_DIR/f"experiment-{M3S3_JOB_ID}.json").write_text(json.dumps({"job_id":M3S3_JOB_ID,"run_dir":str(RUN_DIR),"settings":LAB_SETTINGS,"source_fingerprint":fingerprint,"report":REPORT},indent=2))
    print("LOCAL CORE PASS. These are local measurements, not SciTech evidence.")
    table(["Tool","Actual status"],[(n,t["status"]) for n,t in REPORT["tools"].items()])
else:
    print("No Slurm detected. No benchmark was run. Prediction questions and models still work.")


# 4 — Repetition and fair measurement
**Predict:** Will one outlier change the mean more than the median? Which speedup matters: a function or the whole application?

The internal total spans initialization → compute → reduction → other work. Process wall time also includes launch, validation and cleanup. Keep these boundaries distinct.


In [ ]:
if ready():
    samples=REPORT["measurements"]["locality"]
    table(["Run","Mode","Internal total (s)","Process wall (s)"],
          [(x['run'],x['mode'],f"{float(x['total']['seconds']):.6f}",f"{x['process_wall']:.6f}") for x in samples])
    groups={m:[float(x['total']['seconds']) for x in samples if x['mode']==m] for m in ('bad','good')}
    table(["Mode","Median","Mean","Min","Max"],[(m,*[f"{v:.6f}" for v in (statistics.median(a),statistics.mean(a),min(a),max(a))]) for m,a in groups.items()])
    print("Whole pipeline bad/good median ratio:",round(statistics.median(groups['bad'])/statistics.median(groups['good']),3))
    bar_chart(list(groups),[statistics.median(v) for v in groups.values()],"Same workload, different traversal")


**Change and explain:** change only `N`, rerun from Section 1, and report median plus range for both modes. A smaller matrix may fit into cache. A speedup here supports an access-order hypothesis; it does not independently measure DRAM bandwidth or cache misses.


# 5 — Locate the expensive phase
Timers reveal **where**, not automatically **why**. After improving locality, a different phase may dominate.


In [ ]:
if ready():
    phases=('initialise','compute','reduction','other')
    phase_medians={}
    for mode in ('bad','good'):
        phase_medians[mode]={p:statistics.median([float(r['seconds']) for x in REPORT['measurements']['locality'] if x['mode']==mode for r in x['phases'] if r['phase']==p]) for p in phases}
    table(['Phase','Bad median (s)','Good median (s)'],[(p,f"{phase_medians['bad'][p]:.6f}",f"{phase_medians['good'][p]:.6f}") for p in phases])
    bar_chart(phases,[phase_medians['bad'][p] for p in phases],"Baseline phase timings")
    print("Compute-phase ratio:",phase_medians['bad']['compute']/phase_medians['good']['compute'])


**Try:** increase only `OTHER_ITERS`. The locality optimization remains valid, but its whole-program benefit can shrink. Explain the new critical cost before choosing the next change.


# 6 — GNU time: resources for the application
`/usr/bin/time -v ./profile bad N REPS OTHER_ITERS` runs **inside the compute step**, around the application.

Read elapsed wall time, user/system CPU time, maximum RSS and page faults. CPU time is work charged to the process; wall time also includes delays. Maximum RSS is peak resident memory, not every allocated byte. This single tool run is resource evidence, not the repeated baseline.


In [ ]:
tool_output("gnu_time")


# 7 — gprof: real function profile and call graph
Build: `gcc -O3 -g -pg -fno-omit-frame-pointer -std=c11 ... -lm`. Run the instrumented serial program, then `gprof ./profile_pg gmon.out`.

**Important:** gprof combines instrumented call counts with **sampled CPU-time estimates**. It is not an exact per-call wall timer. Short runs can collect few samples; a zero entry does not prove a function is free. The call graph relates callers and callees. Inclusive time may overlap between parent and child, so do not add it as disjoint phase cost.

**Predict:** Which function should dominate? Can that alone prove memory bandwidth is the bottleneck?


In [ ]:
tool_output("gprof", 90)


**Try:** change `OTHER_ITERS` and compare the hotspot ranking with Section 5. If the profile says “no time accumulated,” increase the work or use region timers; never turn an empty profile into invented samples. Use gprof here for the serial program, not for precise OpenMP waiting attribution.


# 8 — perf: sampling and hardware counters when permitted
The runner attempts `perf stat -e cycles,instructions,cache-references,cache-misses` on both loop orders, then `perf record -g` and `perf report --stdio`.

**Counters are conditional.** Cluster policy or hardware may block them. A denied event is missing evidence, not a zero count. Generic cache events are processor-dependent and do not automatically mean L3 misses or DRAM traffic. A single perf run needs repetition before a strong comparison.

If available: compare instructions, cycles, IPC = instructions/cycles, and misses/accesses where meaningful. A lower IPC is a clue, not proof of a memory bottleneck. Sampling locates hot instruction addresses; counters help test an explanation.


In [ ]:
for name in ('perf_stat_bad','perf_stat_good','perf_report'):
    tool_output(name,45)


# 9 — Scientific output: many writes versus buffered writes
Both programs write the same number of 64-byte records. Small mode writes one record at a time; buffered mode combines records. Both reread the file and check **every byte** and checksum after timing.

The timer includes writes and close; it excludes open/allocation and verification. **No fsync:** this is local filesystem/page-cache/syscall behavior, not sustained durable disk or parallel-filesystem bandwidth.


In [ ]:
if SHOW_C_CODE: show_source('m3s3_syscall_demo.c','main')
if ready():
    io=REPORT['measurements']['io']
    rows=[]
    for mode in ('small','buffered'):
        a=[x for x in io if x['mode']==mode]; vals=[float(x['seconds']) for x in a]
        rows.append((mode,a[0]['bytes'],a[0]['write_calls'],*[f"{v:.6f}" for v in summary(vals)]))
    table(['Mode','Identical bytes','Write calls','Median (s)','Min','Max'],rows)


**Try:** set `BUFFER_RECORDS=1` or increase `RECORDS`. Predict the write count before rerunning. Fewer syscalls are directly observed; a runtime improvement still needs repeated timings. Large production outputs add storage and network effects beyond this small demo.


# 10 — strace: summary versus event sequence
`strace -c` summarizes syscall counts and time charged to traced syscalls. `strace -tt -T` preserves timestamps and syscall durations.

Compare small/buffered **write counts**. Startup/printing/verification also cause syscalls; use the program's own `write_calls` to isolate the output loop. Tracing adds overhead, so compare untraced runtimes from Section 9. A syscall duration does not automatically equal physical disk time.


In [ ]:
for name in ('strace_small','strace_buffered'): tool_output(name,45)
if ready():
    path=RUN_DIR/'strace_events.txt'
    if path.exists(): print('Real timestamped events:\n'+'\n'.join(path.read_text().splitlines()[:20]))
    else: tool_output('strace_events')


# 11 — OpenMP: waiting is a symptom; change ownership
A set of identical independent simulation tasks is executed exactly once. Skewed mode assigns one task to most threads and `SKEW` tasks to the last. Balanced mode distributes **the same tasks** round-robin. All results are checked against a serial reference.

`#pragma omp parallel` creates the worker team; `#pragma omp barrier` waits for every worker. Each worker measures compute and barrier wait. The whole team is timed separately.

**Predict:** Who waits? Does removing the necessary barrier fix unequal ownership? What happens when `SKEW=1`?


In [ ]:
if SHOW_C_CODE: show_source('m3s3_openmp_wait.c','main')
if ready():
    omp=REPORT['measurements']['openmp']; medians=[]
    for mode in ('skewed','balanced'):
        vals=[float(x['total']['seconds']) for x in omp if x['mode']==mode]
        medians.append(statistics.median(vals))
        print(mode, 'whole-team median/min/max:', summary(vals))
        first=next(x for x in omp if x['mode']==mode)
        table(['Thread','Tasks','Compute (s)','Barrier wait (s)'],[(r['thread'],r['tasks'],r['compute'],r['barrier_wait']) for r in first['threads']])
    bar_chart(['skewed','balanced'],medians,'Identical tasks, different ownership')
    print('Skewed/balanced median ratio:',medians[0]/medians[1])


These are **manual region timestamps**, not a full event trace. Sum of thread wait times is not extra elapsed runtime: waits overlap. Critical-path reasoning uses the latest completion, not the average thread.

Below is an interval view from one actual run. Blue = measured computation, orange = measured barrier interval. Timer skew and runtime overhead remain visible; the picture is not fabricated to fit an ideal schedule.


In [ ]:
if ready():
    from IPython.display import SVG, display
    chosen=next(x for x in REPORT['measurements']['openmp'] if x['mode']=='skewed')
    rows=chosen['threads']; scale=max(float(r['compute_end'])+float(r['barrier_wait']) for r in rows) or 1
    svg=[f'<svg xmlns="http://www.w3.org/2000/svg" width="740" height="{70+32*len(rows)}"><rect width="100%" height="100%" fill="white"/><text x="12" y="22">Actual worker intervals: compute (blue), barrier (orange)</text>']
    for i,r in enumerate(rows):
        y=45+i*32; start=float(r['start']); end=float(r['compute_end']); wait=float(r['barrier_wait'])
        svg += [f'<text x="12" y="{y+15}">Thread {r["thread"]}</text>',f'<rect x="{120+560*start/scale}" y="{y}" width="{560*(end-start)/scale}" height="21" fill="#216e9b"/>',f'<rect x="{120+560*end/scale}" y="{y}" width="{560*wait/scale}" height="21" fill="#dc8a24"/>']
    display(SVG(''.join(svg)+'</svg>'))


**Try:** change `SKEW` to 1, then `THREADS` independently. Describe work ownership → completion times → waiting → whole-team runtime. A barrier hotspot can result from upstream load imbalance. Similar reasoning applies to MPI ranks, but this notebook does not measure network communication or multi-node MPI.


# 12 — Score-P: HPC profile and real OTF2 trace
SciTech's batch script attempts its existing EESSI modules: `Score-P/8.4-gompi-2023b` and `CubeLib/4.8.2-GCCcore-13.2.0`. Version names may differ on other clusters; an unavailable module is reported explicitly.

A wrapper builds the **same C sources**. Profile mode creates `profile.cubex`; trace mode must create **`traces.otf2`**, not merely a directory. `scorep-score` estimates trace volume from profile data; it is not itself a timeline viewer. `cube_dump` can summarize function/parallel-region time when installed.

OTF2 files are inspectable in a compatible trace viewer such as Vampir. Creating a trace file does not prove that we have already visualized its events or diagnosed every wait.


In [ ]:
for name in ('scorep','scorep_profile_serial','cube_serial','scorep_profile_openmp','cube_openmp','scorep_trace'):
    if REPORT and name in REPORT['tools']: tool_output(name,35)
if ready():
    table(['Artifact','Relative path'],list(REPORT['artifacts'].items()))
    print('Artifacts remain in:',RUN_DIR)


# 13 — Measurement changes the program
The normal and `-pg` builds use identical optimization flags and inputs, with alternating order and repeated runs. Below compares **process wall times**, including validation, startup, cleanup and gmon output. It is an observed end-to-end instrumentation comparison, not a universal overhead percentage.

Even overhead below 1× can occur in short/noisy measurements. Repeat before interpreting a small difference. Sampling, instrumentation and tracing have workload-dependent costs.


In [ ]:
if ready():
    a=REPORT['measurements']['instrumentation']
    normal=[x['process_wall'] for x in a if x['binary']=='profile']
    profiled=[x['process_wall'] for x in a if x['binary']=='profile_pg']
    table(['Build','Median process wall','Min','Max'],[(b,*[f"{v:.6f}" for v in summary(vals)]) for b,vals in [('normal',normal),('-pg',profiled)]])
    print('Instrumented/normal median process ratio:',statistics.median(profiled)/statistics.median(normal))


# 14 — Model: how much can one hotspot improvement help?
If the compute phase occupies fraction `f` of baseline time and improves by factor `s`, the simple bound is `1 / ((1-f) + f/s)`.

The model assumes other costs stay fixed. Initialization, caches and noise can change, so agreement is not guaranteed. It predicts total-runtime benefit; it is not a new measured experiment.


In [ ]:
MODEL_FRACTION=0.8
MODEL_PHASE_SPEEDUP=10.0
if REPORT:
    local=REPORT['measurements']['locality']
    bad=[x for x in local if x['mode']=='bad']; good=[x for x in local if x['mode']=='good']
    comp=lambda xs: statistics.median([float(r['seconds']) for x in xs for r in x['phases'] if r['phase']=='compute'])
    total=lambda xs:statistics.median([float(x['total']['seconds']) for x in xs])
    MODEL_FRACTION=min(1.0,comp(bad)/total(bad)); MODEL_PHASE_SPEEDUP=comp(bad)/comp(good)
    print('Measured whole-program ratio:',total(bad)/total(good))
print('MODEL compute fraction:',MODEL_FRACTION,'phase speedup:',MODEL_PHASE_SPEEDUP)
print('MODEL predicted whole-program speedup:',1/((1-MODEL_FRACTION)+MODEL_FRACTION/MODEL_PHASE_SPEEDUP))


# 15 — Memory checking: correctness is different from performance
A tiny C program deliberately omits `free` in `leak` mode and releases the identical allocation in `fixed` mode. Valgrind, when available, must actually detect the 16 KiB definite leak. The deliberate leak is confined to a short child process; it is not part of the benchmark pipeline.

`EXPECTED_LEAK_DETECTED` means the tool found the intentionally faulty case. `OK` means the fixed case passed. Neither report diagnoses a cache bottleneck. Static analysis does not execute the program; dynamic checking observes this run. No stale pointer-bounds compiler flags are used.


In [ ]:
if SHOW_C_CODE: show_source('m3s3_memory_check.c','allocation')
for name in ('valgrind_leak','valgrind_fixed','valgrind'):
    if REPORT and name in REPORT['tools']: tool_output(name,30)


# 16 — Performance detective: use your own evidence
For each result below, write **observation → hypothesis → next measurement → proposed change**.

A. A function dominates the profile. Does that distinguish compute throughput, cache latency and memory bandwidth?

B. Threads wait at a barrier. Which earlier thread completion explains it? Would balanced ownership help?

C. Thousands of writes occur. Can buffering reduce calls while keeping bytes and content fixed?

D. Hardware counters were denied. What can you conclude from timers and the access-order change, and what remains unverified?


In [ ]:
if ready():
    print('A: matrix pipeline samples:',len(REPORT['measurements']['locality']))
    print('B: one skewed worker table:')
    x=next(x for x in REPORT['measurements']['openmp'] if x['mode']=='skewed')
    table(['Thread','Compute','Wait'],[(r['thread'],r['compute'],r['barrier_wait']) for r in x['threads']])
    print('C: write counts:',[(x['mode'],x['write_calls']) for x in REPORT['measurements']['io'][:2]])
    print('D: perf status:',REPORT['tools'].get('perf_stat_bad',{}).get('status','not run'))


<details><summary>Suggested reasoning, after students answer</summary>

A hotspot localizes cost; timers alone do not identify the limiting resource. Test access order and, if permitted, counter evidence. In this lab the code shows the access pattern directly.

Barrier waits can be consequences of uneven task ownership; inspect per-thread computation before changing synchronization. Compare balanced mode with the same total tasks and checked results.

Fewer writes can reduce syscall overhead. Keep bytes fixed and compare untraced repeated runtimes; this is not proof of faster physical disks.

Unavailable counters leave a limit on the claim. Report the measured runtime result and the plausible mechanism separately.
</details>


# 17 — Compare two saved experiments and write a claim
After your first run, copy its printed Experiment ID. Change one setting, run again, then enter both IDs below. Records include settings, source fingerprint, compiler flags, CPU set, raw measurements and tool statuses.

**Practice 3 report:** input/resources → baseline median/range → evidence → hypothesis → one change → correctness → new median/range → limitations. Never report only the fastest run.


In [ ]:
BEFORE_ID = None  # e.g. "22454"
AFTER_ID = None
if BEFORE_ID and AFTER_ID:
    records=[json.loads((BASE_DIR/f'experiment-{j}.json').read_text()) for j in (BEFORE_ID,AFTER_ID)]
    changed={k:(records[0]['settings'][k],records[1]['settings'][k]) for k in LAB_SETTINGS if records[0]['settings'][k]!=records[1]['settings'][k]}
    print('Changed settings:',changed)
    if len(changed)!=1: print('This is not a one-setting comparison; explain the additional changes.')
    if records[0]['source_fingerprint']!=records[1]['source_fingerprint']:
        # Fingerprint includes settings; compare actual source hashes separately.
        if records[0]['report']['source_sha256']!=records[1]['report']['source_sha256']: print('Source code also changed.')
    for record in records:
        r=record['report']; print('Experiment',record['job_id'],'host',r['host'],'CPU set',r['allowed_cpus'],'flags',r['compiler_flags'])
        rows=[]
        for family, modes in [('locality',('bad','good')),('openmp',('skewed','balanced')),('io',('small','buffered'))]:
            for mode in modes:
                vals=[float(x.get('total',x)['seconds']) for x in r['measurements'][family] if x['mode']==mode]
                rows.append((family,mode,*[f'{v:.6f}' for v in summary(vals)]))
        table(['Experiment','Mode','Median','Min','Max'],rows)
    print('Different CPU sets or hosts are potential confounders. Compare on the same node type and declared resources.')
else:
    print('Enter two completed Experiment IDs after changing one input. No comparison has been fabricated.')


# 18 — Exit ticket
Answer in one or two sentences:
1. What is the difference between wall time and CPU time?
2. Why repeat and alternate measurement order?
3. Why is a hotspot not automatically a bottleneck?
4. What part of gprof is sampled, and what part is instrumented?
5. Why can a barrier show large waiting when the barrier itself is not the root problem?
6. What evidence proves that buffered output changed calls without changing results?
7. What must you report when perf or Score-P is unavailable?
8. Why check correctness after each optimization?

<details><summary>Concise answers</summary>

Wall time measures elapsed delay; CPU time measures processor work charged to the process. Repetition exposes variability; alternating order reduces systematic order effects. A hotspot localizes cost, while a bottleneck identifies the limit. gprof instruments call counts and samples CPU time. Unequal preceding work causes faster workers to wait. Checked identical bytes/checksums plus write counts isolate the I/O change. Report tool status and limit claims to collected evidence. Faster incorrect output is not an optimization.
</details>

# 19 — What this lab does and does not establish
You have real C/OpenMP measurements, correctness checks, retained raw evidence and testable changes. Hardware counters, profiles and traces are conditional on real tool success. The core lab does not measure GPU occupancy, energy, or durable parallel-filesystem throughput. The optional MPI appendix adds a small two-node communication demo, and the optional GPU appendix measures transfer and kernel time on one allocated GPU. Neither appendix is a full scalability study.

Next: **M3.S4 — Libraries, I/O and reproducible workflows**. Apply the same diagnosis cycle to a fuller application and Practice 3.

References: [GNU gprof](https://sourceware.org/binutils/docs/gprof.html), [perf stat](https://man7.org/linux/man-pages/man1/perf-stat.1.html), [strace](https://man7.org/linux/man-pages/man1/strace.1.html), [Score-P measurement](https://perftools.pages.jsc.fz-juelich.de/cicd/scorep/tags/latest/html/measurement.html), [Valgrind Memcheck](https://valgrind.org/docs/manual/mc-manual.html).


# Appendix — Score-P + MPI: single node and two nodes
This optional extension keeps the existing Session 3 flow unchanged and reuses the **same MPI C program** in two Slurm allocations:

1. **Single node:** 4 MPI ranks on 1 node.
2. **Two nodes:** 4 MPI ranks distributed as 2 ranks per node.

Both runs use Score-P when the SciTech/EESSI module is available. The program deliberately gives the last rank more compute work before an `MPI_Barrier`, so students can see that time inside a barrier may be caused by **load imbalance before the barrier**.

The two-node run also adds repeated cross-rank message exchanges so that Score-P has real inter-node MPI communication to record. This is a teaching experiment, not a full network benchmark.

**Predict before running:** Which ranks should wait longest at the barrier? What changes when ranks are spread across two nodes?


In [ ]:
from pathlib import Path
import subprocess, shutil, textwrap, time

MPI_DEMO = Path.cwd() / "m3s3_scorep_mpi"
MPI_DEMO.mkdir(exist_ok=True)

source = r"""
#include <mpi.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>

static volatile double sink_value = 0.0;

__attribute__((noinline))
static double do_work(uint64_t iters) {
    double x = 1.000001;
    for (uint64_t i = 0; i < iters; ++i)
        x = x * 1.00000001 + 0.00000003;
    return x;
}

int main(int argc, char **argv) {
    MPI_Init(&argc, &argv);

    int rank = 0, size = 0, name_len = 0;
    char host[MPI_MAX_PROCESSOR_NAME];
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    MPI_Get_processor_name(host, &name_len);

    if (size < 2) {
        if (rank == 0) fprintf(stderr, "Run with at least 2 MPI ranks.\n");
        MPI_Finalize();
        return 2;
    }

    const uint64_t base = 2500000ULL;
    const uint64_t iters = base * (rank == size - 1 ? 8ULL : 1ULL);

    double t0 = MPI_Wtime();
    double work = do_work(iters);
    double compute_s = MPI_Wtime() - t0;
    sink_value += work;

    double b0 = MPI_Wtime();
    MPI_Barrier(MPI_COMM_WORLD);
    double barrier_s = MPI_Wtime() - b0;

    /* A small ring exchange repeated enough times to show MPI communication. */
    const int count = 65536;
    const int rounds = 25;
    double *sendbuf = (double*)malloc((size_t)count * sizeof(double));
    double *recvbuf = (double*)malloc((size_t)count * sizeof(double));
    if (!sendbuf || !recvbuf) {
        free(sendbuf); free(recvbuf);
        MPI_Abort(MPI_COMM_WORLD, 3);
    }
    for (int i = 0; i < count; ++i) sendbuf[i] = rank + i * 1e-9;

    int next = (rank + 1) % size;
    int prev = (rank - 1 + size) % size;
    double c0 = MPI_Wtime();
    for (int r = 0; r < rounds; ++r) {
        MPI_Sendrecv(sendbuf, count, MPI_DOUBLE, next, 100,
                     recvbuf, count, MPI_DOUBLE, prev, 100,
                     MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        double *tmp = sendbuf; sendbuf = recvbuf; recvbuf = tmp;
    }
    double comm_s = MPI_Wtime() - c0;

    double local = (double)(rank + 1);
    double global = 0.0;
    double r0 = MPI_Wtime();
    MPI_Reduce(&local, &global, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);
    double reduce_s = MPI_Wtime() - r0;

    printf("MPI_TRACE rank=%d host=%s ranks=%d work_iters=%llu compute=%.6f barrier_wait=%.6f ring_comm=%.6f reduce=%.6f\n",
           rank, host, size, (unsigned long long)iters,
           compute_s, barrier_s, comm_s, reduce_s);

    int ok = 1;
    if (rank == 0) {
        double expected = (double)size * (size + 1) / 2.0;
        ok = fabs(global - expected) < 1e-12;
        printf("MPI_TOTAL ranks=%d reduced_sum=%.1f expected=%.1f validation=%s\n",
               size, global, expected, ok ? "PASS" : "FAIL");
    }
    MPI_Bcast(&ok, 1, MPI_INT, 0, MPI_COMM_WORLD);

    free(sendbuf); free(recvbuf);
    MPI_Finalize();
    return ok ? 0 : 4;
}
"""
(MPI_DEMO / "mpi_scorep_demo.c").write_text(source)

common = r"""
set -u
cd "$SLURM_SUBMIT_DIR"

EESSI_INIT=/cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash
if [ -r "$EESSI_INIT" ]; then
    source "$EESSI_INIT"
    module load foss/2023b >/dev/null 2>&1 || true
    module load Score-P/8.4-gompi-2023b >/dev/null 2>&1 || true
    module load CubeLib/4.8.2-GCCcore-13.2.0 >/dev/null 2>&1 || true
fi

for tool in scorep mpicc srun; do
    if ! command -v "$tool" >/dev/null 2>&1; then
        echo "MPI_SCOREP_STATUS=UNAVAILABLE missing=$tool"
        exit 0
    fi
done

echo "Nodes allocated:"
scontrol show hostnames "$SLURM_JOB_NODELIST"
echo "Score-P: $(scorep --version 2>/dev/null | head -1)"

scorep --mpp=mpi mpicc -O2 -g mpi_scorep_demo.c -lm -o mpi_scorep_demo
if [ $? -ne 0 ]; then
    echo "MPI_SCOREP_STATUS=COMPILE_FAILED"
    exit 1
fi
"""

single = """#!/bin/bash
#SBATCH --job-name=m3s3_mpi_1node
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G
#SBATCH --time=00:05:00
#SBATCH --output=mpi-1node-%j.out
""" + common + r"""
rm -rf scorep_mpi_1node_profile scorep_mpi_1node_trace

export SCOREP_EXPERIMENT_DIRECTORY="$PWD/scorep_mpi_1node_profile"
export SCOREP_OVERWRITE_EXPERIMENT_DIRECTORY=true
export SCOREP_ENABLE_PROFILING=true
export SCOREP_ENABLE_TRACING=false

echo "=== SINGLE-NODE PROFILE: 4 ranks on 1 node ==="
srun --ntasks=4 ./mpi_scorep_demo
RC=$?
[ $RC -eq 0 ] || exit $RC

if [ -f scorep_mpi_1node_profile/profile.cubex ]; then
    echo "MPI_SCOREP_PROFILE=scorep_mpi_1node_profile/profile.cubex"
    command -v scorep-score >/dev/null 2>&1 && scorep-score scorep_mpi_1node_profile/profile.cubex | head -60 || true
else
    echo "MPI_SCOREP_STATUS=PROFILE_MISSING"
fi

export SCOREP_EXPERIMENT_DIRECTORY="$PWD/scorep_mpi_1node_trace"
export SCOREP_ENABLE_PROFILING=false
export SCOREP_ENABLE_TRACING=true
export SCOREP_TOTAL_MEMORY=64M

echo "=== SINGLE-NODE TRACE ==="
srun --ntasks=4 ./mpi_scorep_demo
RC=$?
[ $RC -eq 0 ] || exit $RC

[ -e scorep_mpi_1node_trace/traces.otf2 ] && echo "MPI_SCOREP_TRACE=scorep_mpi_1node_trace/traces.otf2" || echo "MPI_SCOREP_STATUS=TRACE_MISSING"
echo "MPI_SCOREP_STATUS=PASS"
"""

multi = """#!/bin/bash
#SBATCH --job-name=m3s3_mpi_2nodes
#SBATCH --partition=cpu
#SBATCH --nodes=2
#SBATCH --ntasks=4
#SBATCH --ntasks-per-node=2
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G
#SBATCH --time=00:05:00
#SBATCH --output=mpi-2nodes-%j.out
""" + common + r"""
rm -rf scorep_mpi_2nodes_profile scorep_mpi_2nodes_trace

export SCOREP_EXPERIMENT_DIRECTORY="$PWD/scorep_mpi_2nodes_profile"
export SCOREP_OVERWRITE_EXPERIMENT_DIRECTORY=true
export SCOREP_ENABLE_PROFILING=true
export SCOREP_ENABLE_TRACING=false

echo "=== TWO-NODE PROFILE: 4 ranks, 2 ranks per node ==="
srun --nodes=2 --ntasks=4 --ntasks-per-node=2 ./mpi_scorep_demo
RC=$?
[ $RC -eq 0 ] || exit $RC

if [ -f scorep_mpi_2nodes_profile/profile.cubex ]; then
    echo "MPI_SCOREP_PROFILE=scorep_mpi_2nodes_profile/profile.cubex"
    command -v scorep-score >/dev/null 2>&1 && scorep-score scorep_mpi_2nodes_profile/profile.cubex | head -60 || true
else
    echo "MPI_SCOREP_STATUS=PROFILE_MISSING"
fi

export SCOREP_EXPERIMENT_DIRECTORY="$PWD/scorep_mpi_2nodes_trace"
export SCOREP_ENABLE_PROFILING=false
export SCOREP_ENABLE_TRACING=true
export SCOREP_TOTAL_MEMORY=96M

echo "=== TWO-NODE TRACE ==="
srun --nodes=2 --ntasks=4 --ntasks-per-node=2 ./mpi_scorep_demo
RC=$?
[ $RC -eq 0 ] || exit $RC

[ -e scorep_mpi_2nodes_trace/traces.otf2 ] && echo "MPI_SCOREP_TRACE=scorep_mpi_2nodes_trace/traces.otf2" || echo "MPI_SCOREP_STATUS=TRACE_MISSING"
echo "MPI_SCOREP_STATUS=PASS"
"""

(MPI_DEMO / "run_scorep_mpi_1node.sbatch").write_text(single)
(MPI_DEMO / "run_scorep_mpi_2nodes.sbatch").write_text(multi)

print("Created:")
for p in sorted(MPI_DEMO.iterdir()):
    print(" ", p.name)
print("\nRun the next two cells to submit the single-node and two-node experiments.")


In [ ]:
# Single-node MPI + Score-P: 4 ranks on 1 node
if shutil.which("sbatch") is None:
    print("UNAVAILABLE: sbatch is not on this host.")
else:
    p = subprocess.run(
        ["sbatch", "--wait", "--parsable", "run_scorep_mpi_1node.sbatch"],
        cwd=MPI_DEMO, capture_output=True, text=True, timeout=240
    )
    print("sbatch rc:", p.returncode, "job:", p.stdout.strip())
    if p.stderr.strip(): print(p.stderr.strip())
    logs = sorted(MPI_DEMO.glob("mpi-1node-*.out"), key=lambda x: x.stat().st_mtime)
    if logs:
        print(logs[-1].read_text()[:14000])
        print("Full log:", logs[-1])


In [ ]:
# Two-node MPI + Score-P: 4 ranks across 2 nodes
# This may wait longer in the Slurm queue because it requires two nodes at once.
if shutil.which("sbatch") is None:
    print("UNAVAILABLE: sbatch is not on this host.")
else:
    p = subprocess.run(
        ["sbatch", "--wait", "--parsable", "run_scorep_mpi_2nodes.sbatch"],
        cwd=MPI_DEMO, capture_output=True, text=True, timeout=360
    )
    print("sbatch rc:", p.returncode, "job:", p.stdout.strip())
    if p.stderr.strip(): print(p.stderr.strip())
    logs = sorted(MPI_DEMO.glob("mpi-2nodes-*.out"), key=lambda x: x.stat().st_mtime)
    if logs:
        print(logs[-1].read_text()[:16000])
        print("Full log:", logs[-1])


## What students should compare

For the **single-node run**, all four ranks share one compute node. For the **two-node run**, Slurm places two ranks on each of two nodes.

Compare:

- the printed `host=` field for every rank;
- each rank's `compute` time;
- `barrier_wait`: early ranks should wait for the deliberately slower rank;
- `ring_comm`: on two nodes, some messages cross the interconnect rather than remaining within one node;
- the Score-P `profile.cubex` files for time attributed to MPI calls;
- the OTF2 traces for the event ordering across ranks.

**Important:** a two-node run introduces real network communication, but one short run is **not** enough to claim a network bandwidth or scaling result. Repeat measurements and keep the allocation, message size, rank placement, and executable fixed before comparing performance.

**Try:** change the imbalance multiplier from `8ULL` to `1ULL`. Then rerun both allocations. If barrier waiting falls, the evidence supports the hypothesis that the barrier hotspot was caused by upstream load imbalance.


# Appendix — GPU profiling: transfers and kernel time
This optional exercise uses the same GPU allocation model as M2.S4: one Slurm job on the `gpu` partition with one GPU.

It answers two questions:

1. **Which GPU tools are actually available on the allocated node?** The job checks `nvidia-smi`, `nvcc`, `nsys` and `ncu`.
2. **Where does a simple CUDA operation spend its time?** A vector-add program measures host→GPU transfer, kernel execution and GPU→host transfer separately.

If CUDA or Nsight is unavailable, the job reports that status and stops cleanly. No GPU measurement is inferred from the Jupyter host.

**Predict before running:** for a simple vector addition, will data movement take more or less time than the kernel?


In [ ]:
GPU_DEMO = Path.cwd() / "m3s3_gpu_profile"
GPU_DEMO.mkdir(exist_ok=True)

cuda_source = r'''
#include <cuda_runtime.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>

#define CUDA_CHECK(call) do { \
    cudaError_t e = (call); \
    if (e != cudaSuccess) { \
        fprintf(stderr, "CUDA error: %s at %s:%d\n", cudaGetErrorString(e), __FILE__, __LINE__); \
        return 2; \
    } \
} while (0)

__global__ void vector_add(const float *a, const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main(void) {
    const int n = 1 << 20;
    const size_t bytes = (size_t)n * sizeof(float);

    float *a = (float*)malloc(bytes);
    float *b = (float*)malloc(bytes);
    float *c = (float*)malloc(bytes);
    if (!a || !b || !c) {
        fprintf(stderr, "Host allocation failed\n");
        return 2;
    }

    for (int i = 0; i < n; ++i) {
        a[i] = 0.5f + (float)(i % 100) * 0.001f;
        b[i] = 1.5f - (float)(i % 100) * 0.001f;
    }

    float *d_a = NULL, *d_b = NULL, *d_c = NULL;
    CUDA_CHECK(cudaMalloc((void**)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_c, bytes));

    /* Initialise the CUDA context before the timed regions. */
    CUDA_CHECK(cudaFree(0));

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    float h2d_ms = 0.0f, kernel_ms = 0.0f, d2h_ms = 0.0f;

    CUDA_CHECK(cudaEventRecord(start));
    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&h2d_ms, start, stop));

    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;

    CUDA_CHECK(cudaEventRecord(start));
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&kernel_ms, start, stop));

    CUDA_CHECK(cudaEventRecord(start));
    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&d2h_ms, start, stop));

    int ok = 1;
    for (int i = 0; i < n; ++i) {
        float expected = a[i] + b[i];
        if (fabsf(c[i] - expected) > 1e-6f) {
            ok = 0;
            break;
        }
    }

    printf("GPU_TIMING elements=%d MiB_each=%.2f h2d_ms=%.6f kernel_ms=%.6f d2h_ms=%.6f measured_total_ms=%.6f validation=%s\n",
           n, bytes / 1048576.0, h2d_ms, kernel_ms, d2h_ms,
           h2d_ms + kernel_ms + d2h_ms, ok ? "PASS" : "FAIL");

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    free(a); free(b); free(c);

    return ok ? 0 : 3;
}
'''
(GPU_DEMO / "gpu_profile_demo.cu").write_text(cuda_source)

gpu_job = r'''#!/bin/bash
#SBATCH --job-name=m3s3_gpu_profile
#SBATCH --partition=gpu
#SBATCH --gpus=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=4G
#SBATCH --time=00:05:00
#SBATCH --output=gpu-profile-%j.out

set -u
cd "$SLURM_SUBMIT_DIR"

unset SLURM_MEM_PER_CPU
unset SLURM_MEM_PER_GPU
unset SLURM_MEM_PER_NODE

echo "=== GPU NODE ==="
echo "Host=$(hostname)"
if command -v nvidia-smi >/dev/null 2>&1; then
    nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
else
    echo "GPU_TOOL nvidia-smi=UNAVAILABLE"
fi

echo
echo "=== LOAD CUDA TOOLCHAIN ==="
module purge >/dev/null 2>&1 || true
if module load nvhpc/25.7 >/dev/null 2>&1; then
    echo "GPU_MODULE nvhpc/25.7=OK"
elif module load hpc-course-cuda/12.8.1 >/dev/null 2>&1; then
    echo "GPU_MODULE hpc-course-cuda/12.8.1=OK"
else
    echo "GPU_MODULE=UNAVAILABLE"
fi

echo
echo "=== TOOL CHECK ==="
for tool in nvidia-smi nvcc nsys ncu; do
    if command -v "$tool" >/dev/null 2>&1; then
        echo "GPU_TOOL $tool=OK path=$(command -v "$tool")"
    else
        echo "GPU_TOOL $tool=UNAVAILABLE"
    fi
done

if ! command -v nvcc >/dev/null 2>&1; then
    echo "GPU_EXERCISE_STATUS=UNAVAILABLE reason=nvcc_not_found"
    exit 0
fi

echo
echo "=== CUDA TIMING ==="
nvcc -O3 -Wno-deprecated-gpu-targets gpu_profile_demo.cu -o gpu_profile_demo
if [ $? -ne 0 ]; then
    echo "GPU_EXERCISE_STATUS=COMPILE_FAILED"
    exit 1
fi

./gpu_profile_demo
RC=$?
if [ $RC -ne 0 ]; then
    echo "GPU_EXERCISE_STATUS=RUN_FAILED rc=$RC"
    exit $RC
fi

echo
echo "=== NSIGHT SYSTEMS ==="
if command -v nsys >/dev/null 2>&1; then
    rm -f m3s3_gpu_nsys.nsys-rep m3s3_gpu_nsys.qdrep
    nsys profile --force-overwrite=true --sample=none --trace=cuda -o m3s3_gpu_nsys ./gpu_profile_demo
    NSYS_RC=$?
    if [ $NSYS_RC -eq 0 ]; then
        echo "NSYS_PROFILE=OK"
        if [ -f m3s3_gpu_nsys.nsys-rep ]; then
            echo "NSYS_ARTIFACT=m3s3_gpu_nsys.nsys-rep"
            nsys stats m3s3_gpu_nsys.nsys-rep 2>/dev/null | head -120 || true
        elif [ -f m3s3_gpu_nsys.qdrep ]; then
            echo "NSYS_ARTIFACT=m3s3_gpu_nsys.qdrep"
        fi
    else
        echo "NSYS_PROFILE=UNAVAILABLE_OR_RESTRICTED rc=$NSYS_RC"
    fi
else
    echo "NSYS_PROFILE=UNAVAILABLE"
fi

echo
echo "=== NSIGHT COMPUTE ==="
if command -v ncu >/dev/null 2>&1; then
    echo "NCU_STATUS=AVAILABLE"
    ncu --set basic --target-processes all ./gpu_profile_demo > ncu-basic.txt 2>&1
    NCU_RC=$?
    head -120 ncu-basic.txt
    if [ $NCU_RC -eq 0 ]; then
        echo "NCU_RUN=OK"
    else
        echo "NCU_RUN=UNAVAILABLE_OR_RESTRICTED rc=$NCU_RC"
    fi
else
    echo "NCU_STATUS=UNAVAILABLE"
fi

echo "GPU_EXERCISE_STATUS=PASS"
'''
(GPU_DEMO / "run_gpu_profile.sbatch").write_text(gpu_job)

print("Created:")
print(" ", GPU_DEMO / "gpu_profile_demo.cu")
print(" ", GPU_DEMO / "run_gpu_profile.sbatch")
print("\nThe next cell submits one short GPU job.")


In [ ]:
# Submit the optional GPU profiling exercise.
if shutil.which("sbatch") is None:
    print("GPU exercise unavailable here: sbatch is not on this host.")
else:
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key, None)

    p = subprocess.run(
        ["sbatch", "--wait", "--parsable", "run_gpu_profile.sbatch"],
        cwd=GPU_DEMO,
        capture_output=True,
        text=True,
        env=env,
        timeout=360,
    )
    print("sbatch rc:", p.returncode, "job:", p.stdout.strip())
    if p.stderr.strip():
        print(p.stderr.strip())

    logs = sorted(GPU_DEMO.glob("gpu-profile-*.out"), key=lambda x: x.stat().st_mtime)
    if logs:
        GPU_PROFILE_LOG = logs[-1].read_text(errors="replace")
        print(GPU_PROFILE_LOG[:20000])
        print("Full log:", logs[-1])
    else:
        GPU_PROFILE_LOG = ""
        print("No Slurm output file found.")


## Read the result

The line beginning with `GPU_TIMING` separates three measured phases:

- `h2d_ms`: copy the two input arrays from host memory to GPU memory;
- `kernel_ms`: run the CUDA vector-add kernel;
- `d2h_ms`: copy the result back to host memory.

The program also validates the result. Treat timings as evidence only when `validation=PASS`.

If `nsys` is available, the job also creates a small Nsight Systems report. Use it to confirm that the timeline contains the two host-to-device copies, the kernel launch and the device-to-host copy. If `ncu` is available, the job attempts a basic kernel profile; cluster policy may still restrict hardware metrics.

### Exercise

1. Which phase is largest: host→GPU transfer, kernel execution, or GPU→host transfer?
2. Compute the transfer share:
   [
   rac{	ext{H2D}+	ext{D2H}}{	ext{H2D}+	ext{kernel}+	ext{D2H}}
   ]
3. Change only `n` from `1 << 20` to `1 << 18`, rerun, and explain how the balance changes.
4. Then try `1 << 22`. Does a larger problem make the kernel a larger fraction of the measured GPU work?

Repeat measurements before making a performance claim. A profiler timeline helps explain **where the time goes**; it does not by itself prove that a code change is faster.
